# Ejemplo de uso del modulo con dataset EOD

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "EOD_STGO" 

In [2]:
import pandas as pd
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

In [3]:
eod_viajes = pd.read_csv(DATA_PATH / "Viajes.csv", sep=";", dtype=str, low_memory=False)
eod_personas = pd.read_csv(DATA_PATH / "Personas.csv", sep=";", dtype=str, low_memory=False)
eod_hogares = pd.read_csv(DATA_PATH / "Hogares.csv", sep=";", dtype=str, low_memory=False)

In [4]:
#df_eod_trips_for_import.head(1000).to_csv(DATA_PATH / "temp" / "preproccesed_trips.csv", sep=";", index=False)
#df_eod_stages_for_import.head(1000).to_csv(DATA_PATH / "temp" / "preproccesed_stages.csv", sep=";", index=False)

In [5]:
from pathlib import Path
import pandas as pd

from pylondrina.importing import ImportOptions, import_trips_from_dataframe
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec
from pylondrina.transforms.spatial import project_xy_to_latlon

# ------------------------------------------------------------
# 1) Carga de datos
# ------------------------------------------------------------
AUX_PATH = DATA_PATH / "Tablas_parametros"

eod_viajes = pd.read_csv(DATA_PATH / "Viajes.csv", sep=";", dtype=str, low_memory=False)
eod_hogares = pd.read_csv(DATA_PATH / "Hogares.csv", sep=";", dtype=str, low_memory=False)

# En este caso simplificado quiero trabajar sobre la tabla de viajes
# y solo traer lo mínimo para poder importarla.
df_eod_trips_lvl1 = eod_viajes.copy()

# También traigo TipoDia desde Hogares porque es útil para el ejemplo.
df_eod_trips_lvl1 = df_eod_trips_lvl1.merge(
    eod_hogares[["Hogar", "TipoDia"]].copy(),
    on="Hogar",
    how="left",
)

# ------------------------------------------------------------
# 2) Carga de tablas auxiliares necesarias
#    Solo unas pocas, para no sobrecargar el preprocess.
# ------------------------------------------------------------
aux_comuna = pd.read_csv(AUX_PATH / "Comuna.csv", sep=",", dtype=str, low_memory=False)
aux_proposito = pd.read_csv(AUX_PATH / "Proposito.csv", sep=";", dtype=str, low_memory=False)
aux_modo_agregado = pd.read_csv(AUX_PATH / "ModoAgregado.csv", sep=";", dtype=str, low_memory=False)
aux_codigo_tiempo = pd.read_csv(AUX_PATH / "CodigoTiempo.csv", sep=";", dtype=str, low_memory=False, encoding = "latin-1")
aux_tipo_dia = pd.read_csv(AUX_PATH / "TipoDia.csv", sep=";", dtype=str, low_memory=False)

# Para simplificar, asumo que la primera columna es código y la segunda es nombre.
comuna_map = dict(zip(aux_comuna.iloc[:, 0].astype(str), aux_comuna.iloc[:, 1].astype(str)))
proposito_map = dict(zip(aux_proposito.iloc[:, 0].astype(str), aux_proposito.iloc[:, 1].astype(str)))
modo_agregado_map = dict(zip(aux_modo_agregado.iloc[:, 0].astype(str), aux_modo_agregado.iloc[:, 1].astype(str)))
codigo_tiempo_map = dict(zip(aux_codigo_tiempo.iloc[:, 0].astype(str), aux_codigo_tiempo.iloc[:, 1].astype(str)))
tipo_dia_map = dict(zip(aux_tipo_dia.iloc[:, 0].astype(str), aux_tipo_dia.iloc[:, 1].astype(str)))

# ------------------------------------------------------------
# 3) Decodificación mínima de algunos campos categóricos
#    Aquí solo traigo 5 campos desde tablas auxiliares.
# ------------------------------------------------------------
df_eod_trips_lvl1["ComunaOrigen_nombre"] = df_eod_trips_lvl1["ComunaOrigen"].astype(str).map(comuna_map)
df_eod_trips_lvl1["ComunaDestino_nombre"] = df_eod_trips_lvl1["ComunaDestino"].astype(str).map(comuna_map)
df_eod_trips_lvl1["Proposito_nombre"] = df_eod_trips_lvl1["Proposito"].astype(str).map(proposito_map)
df_eod_trips_lvl1["ModoAgregado_nombre"] = df_eod_trips_lvl1["ModoAgregado"].astype(str).map(modo_agregado_map)
df_eod_trips_lvl1["TipoDia_nombre"] = df_eod_trips_lvl1["TipoDia"].astype(str).map(tipo_dia_map)

# Decodifico también CódigoTiempo porque me sirve como ejemplo de campo con dominio abierto.
df_eod_trips_lvl1["CodigoTiempo_nombre"] = df_eod_trips_lvl1["CodigoTiempo"].astype(str).map(codigo_tiempo_map)

# Como dijiste, otros campos categóricos que también requerirían auxiliares
# y que no quiero tratar en este caso, simplemente se eliminan.
cols_to_drop_if_exist = [
    "SectorOrigen",
    "SectorDestino",
    "PropositoAgregado",
    "ActividadDestino",
    "ModoPriPub",
    "ModoMotor",
    "Periodo",
    "TiempoMedio",
]
df_eod_trips_lvl1 = df_eod_trips_lvl1.drop(
    columns=[c for c in cols_to_drop_if_exist if c in df_eod_trips_lvl1.columns],
    errors="ignore",
)

# ------------------------------------------------------------
# 4) Conversión de coordenadas proyectadas a lon/lat
#    Esta es la ayuda puntual del módulo que sí vale la pena usar.
# ------------------------------------------------------------
df_eod_trips_lvl1 = project_xy_to_latlon(
    df_eod_trips_lvl1,
    x_col="OrigenCoordX",
    y_col="OrigenCoordY",
    source_crs="EPSG:5361",
    lon_col="origin_longitude",
    lat_col="origin_latitude",
    keep_debug_cols=False,
    drop_input_cols=False,
)

df_eod_trips_lvl1 = project_xy_to_latlon(
    df_eod_trips_lvl1,
    x_col="DestinoCoordX",
    y_col="DestinoCoordY",
    source_crs="EPSG:5361",
    lon_col="destination_longitude",
    lat_col="destination_latitude",
    keep_debug_cols=False,
    drop_input_cols=False,
)

# ------------------------------------------------------------
# 5) Me quedo solo con columnas mínimas y algunas de contexto
#    No quiero preservar todo el dataset.
# ------------------------------------------------------------
cols_for_import_lvl1 = [
    "Viaje",
    "Persona",
    "HoraIni",
    "HoraFin",
    "origin_longitude",
    "origin_latitude",
    "destination_longitude",
    "destination_latitude",
    "ComunaOrigen_nombre",
    "ComunaDestino_nombre",
    "Proposito_nombre",
    "ModoAgregado_nombre",
    "TipoDia_nombre",
    "CodigoTiempo_nombre",
]

df_eod_trips_lvl1 = df_eod_trips_lvl1[cols_for_import_lvl1].copy()

display(df_eod_trips_lvl1.head())

,Viaje,Persona,HoraIni,HoraFin,origin_longitude,origin_latitude,destination_longitude,destination_latitude,ComunaOrigen_nombre,ComunaDestino_nombre,Proposito_nombre,ModoAgregado_nombre,TipoDia_nombre,CodigoTiempo_nombre
0,1734310202,17343102,22:30,23:40,-70.774666,-33.531422,-70.735153,-33.495874,MAIPÚ,MAIPÚ,volver a casa,Bus TS,Laboral,Tiempo de viaje correcto
1,1734410101,17344101,13:00,14:45,-70.738205,-33.500007,-70.567232,-33.408776,MAIPÚ,LAS CONDES,Al trabajo,Bus TS - Metro,Laboral,Tiempo de viaje correcto
2,1734410102,17344101,22:00,23:30,-70.567232,-33.408776,-70.738205,-33.500007,LAS CONDES,MAIPÚ,volver a casa,Bus TS - Metro,Laboral,Tiempo de viaje correcto
3,1734410301,17344103,9:00,9:55,-70.738205,-33.500007,-70.604904,-33.454153,MAIPÚ,ÑUÑOA,Al trabajo,Bus TS - Metro,Laboral,Tiempo de viaje correcto
4,1734410302,17344103,19:00,21:30,-70.604904,-33.454153,-70.738205,-33.500007,ÑUÑOA,MAIPÚ,volver a casa,Bus TS - Metro,Laboral,Tiempo de viaje correcto


In [21]:
EOD_TRIPS_LVL1_SCHEMA = TripSchema(
    version="eod-trips-lvl1-simplificado",
    fields={
        # Mantengo la mayoría de nombres "cercanos" a lo crudo/preprocesado
        "movement_id": FieldSpec("movement_id", "string", required=True), # Solo cambio el nombre del campo Viajes, para que sea el indice del dataframe
        "Persona": FieldSpec("Persona", "string", required=True),
        "HoraIni": FieldSpec("HoraIni", "string", required=False),
        "HoraFin": FieldSpec("HoraFin", "string", required=False),

        # Estas sí las dejo canónicas porque el import las necesita bien.
        "origin_longitude": FieldSpec("origin_longitude", "float", required=True),
        "origin_latitude": FieldSpec("origin_latitude", "float", required=True),
        "destination_longitude": FieldSpec("destination_longitude", "float", required=True),
        "destination_latitude": FieldSpec("destination_latitude", "float", required=True),

        # Campos decodificados visibles
        "ComunaOrigen": FieldSpec(
            "ComunaOrigen",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=[],          # dominio abierto, se extiende en import
                extendable=True,
            ),
        ),
        "ComunaDestino": FieldSpec(
            "ComunaDestino",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=[],          # dominio abierto, se extiende en import
                extendable=True,
            ),
        ),
        "Proposito": FieldSpec(
            "Proposito",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["Volver a casa", "Al trabajo", "Por trabajo", "Al estudio", "Otra actividad"],
                extendable=False,   # si aparece algo fuera, se va a unknown
            ),
        ),
        "ModoAgregado": FieldSpec(
            "ModoAgregado",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["Auto", "Bus TS", "Bus no TS", "Metro", "Taxi", "Caminata", "Bicicleta", "Otros"],
                extendable=False,
            ),
        ),
        "TipoDia": FieldSpec(
            "TipoDia",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["Laboral", "Fin de Semana"],
                extendable=False,
            ),
        ),
        "CodigoTiempo": FieldSpec(
            "CodigoTiempo",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=[],          # dominio abierto para mostrar extensión
                extendable=True,
            ),
        ),
    },
    required=[
        "movement_id",
        "Persona",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
    ],
)

EOD_TRIPS_LVL1_OPTIONS = ImportOptions(
    keep_extra_fields=False,
    selected_fields=None,
    strict=False,
    strict_domains=False,   # permite extender dominios abiertos
    single_stage=True,      # trip_id y movement_seq se derivan desde movement_id
    source_timezone=None,
)

# Solo unas pocas correspondencias de campos.
# La idea es que casi todo lo demás quede "como viene" en el dataframe.
EOD_TRIPS_LVL1_FIELD_CORRESPONDENCE = {
    "movement_id": "Viaje",
    "CodigoTiempo": "CodigoTiempo_nombre",
    "TipoDia": "TipoDia_nombre",
    "ModoAgregado": "ModoAgregado_nombre",
    "Proposito": "Proposito_nombre",
    "ComunaDestino": "ComunaDestino_nombre",
    "ComunaOrigen": "ComunaOrigen_nombre",
    
}


EOD_TRIPS_LVL1_PROVENANCE = {
    "source": {
        "name": "EOD",
        "entity": "trips",
        "version": "EOD_STGO",
    },
    "notes": [
        "nivel 1 simplificado",
        "solo decodificación mínima de 5-6 campos",
        "uso del helper espacial del módulo",
    ],
}

In [22]:
trips_eod_lvl1, report_eod_lvl1 = import_trips_from_dataframe(
    df_eod_trips_lvl1.head(1000),
    schema=EOD_TRIPS_LVL1_SCHEMA,
    source_name="EOD trips - nivel 1 simplificado",
    options=EOD_TRIPS_LVL1_OPTIONS,
    field_correspondence=EOD_TRIPS_LVL1_FIELD_CORRESPONDENCE,
    provenance=EOD_TRIPS_LVL1_PROVENANCE,
    h3_resolution=8,
)

display(trips_eod_lvl1.data.head())
display(report_eod_lvl1.issues[:10])

,movement_id,trip_id,movement_seq,Persona,HoraIni,HoraFin,origin_longitude,origin_latitude,destination_longitude,destination_latitude,ComunaOrigen,ComunaDestino,Proposito,ModoAgregado,TipoDia,CodigoTiempo
0,1734310202,1734310202,0,17343102,22:30,23:40,-70.774666,-33.531422,-70.735153,-33.495874,MAIPÚ,MAIPÚ,unknown,Bus TS,Laboral,Tiempo de viaje correcto
1,1734410101,1734410101,0,17344101,13:00,14:45,-70.738205,-33.500007,-70.567232,-33.408776,MAIPÚ,LAS CONDES,Al trabajo,unknown,Laboral,Tiempo de viaje correcto
2,1734410102,1734410102,0,17344101,22:00,23:30,-70.567232,-33.408776,-70.738205,-33.500007,LAS CONDES,MAIPÚ,unknown,unknown,Laboral,Tiempo de viaje correcto
3,1734410301,1734410301,0,17344103,9:00,9:55,-70.738205,-33.500007,-70.604904,-33.454153,MAIPÚ,ÑUÑOA,Al trabajo,unknown,Laboral,Tiempo de viaje correcto
4,1734410302,1734410302,0,17344103,19:00,21:30,-70.604904,-33.454153,-70.738205,-33.500007,ÑUÑOA,MAIPÚ,unknown,unknown,Laboral,Tiempo de viaje correcto


[Issue(level='info', code='SCH.DOMAIN.EMPTY_VALUES', message="El DomainSpec del campo 'ComunaOrigen' tiene values vacío; se tratará como dominio bootstrap/extensible según política.", field='ComunaOrigen', source_field=None, row_count=None, details={'field': 'ComunaOrigen', 'values_size': 0, 'extendable': True, 'note': 'candidate_bootstrap'}),
 Issue(level='info', code='SCH.DOMAIN.EMPTY_VALUES', message="El DomainSpec del campo 'ComunaDestino' tiene values vacío; se tratará como dominio bootstrap/extensible según política.", field='ComunaDestino', source_field=None, row_count=None, details={'field': 'ComunaDestino', 'values_size': 0, 'extendable': True, 'note': 'candidate_bootstrap'}),
 Issue(level='info', code='SCH.DOMAIN.EMPTY_VALUES', message="El DomainSpec del campo 'CodigoTiempo' tiene values vacío; se tratará como dominio bootstrap/extensible según política.", field='CodigoTiempo', source_field=None, row_count=None, details={'field': 'CodigoTiempo', 'values_size': 0, 'extendable'

# Nivel 2

### Carga de datos

In [27]:
from pathlib import Path
import pandas as pd

from pylondrina.importing import ImportOptions
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec
from pylondrina.sources.profile import SourceProfile
from pylondrina.sources.helpers import import_trips_from_profile
from pylondrina.transforms.spatial import project_xy_to_latlon

eod_etapas = pd.read_csv(DATA_PATH / "Etapas.csv", sep=";", dtype=str, low_memory=False, encoding="latin-1")
eod_personas = pd.read_csv(DATA_PATH / "Personas.csv", sep=";", dtype=str, low_memory=False)

# ------------------------------------------------------------
# Carga mínima de tablas auxiliares que sí quiero usar en este caso
# ------------------------------------------------------------
aux_comuna = pd.read_csv(AUX_PATH / "Comuna.csv", sep=",", dtype=str, low_memory=False)
aux_modo = pd.read_csv(AUX_PATH / "Modo.csv", sep=";", dtype=str, low_memory=False)
aux_forma_pago = pd.read_csv(AUX_PATH / "Formapago.csv", sep=";", dtype=str, low_memory=False, encoding = "latin-1")
aux_sexo = pd.read_csv(AUX_PATH / "Sexo.csv", sep=";", dtype=str, low_memory=False)
aux_actividad = pd.read_csv(AUX_PATH / "Actividad.csv", sep=";", dtype=str, low_memory=False)
aux_ocupacion = pd.read_csv(AUX_PATH / "Ocupacion.csv", sep=";", dtype=str, low_memory=False)
aux_licencia = pd.read_csv(AUX_PATH / "LicenciaConducir.csv", sep=";", dtype=str, low_memory=False, encoding = "latin-1")
aux_pase = pd.read_csv(AUX_PATH / "PaseEscolar.csv", sep=";", dtype=str, low_memory=False, encoding = "latin-1")

comuna_map = dict(zip(aux_comuna.iloc[:, 0].astype(str), aux_comuna.iloc[:, 1].astype(str)))
modo_map = dict(zip(aux_modo.iloc[:, 0].astype(str), aux_modo.iloc[:, 1].astype(str)))
forma_pago_map = dict(zip(aux_forma_pago.iloc[:, 0].astype(str), aux_forma_pago.iloc[:, 1].astype(str)))

sexo_map = dict(zip(aux_sexo.iloc[:, 0].astype(str), aux_sexo.iloc[:, 1].astype(str)))
actividad_map = dict(zip(aux_actividad.iloc[:, 0].astype(str), aux_actividad.iloc[:, 1].astype(str)))
ocupacion_map = dict(zip(aux_ocupacion.iloc[:, 0].astype(str), aux_ocupacion.iloc[:, 1].astype(str)))
licencia_map = dict(zip(aux_licencia.iloc[:, 0].astype(str), aux_licencia.iloc[:, 1].astype(str)))
pase_map = dict(zip(aux_pase.iloc[:, 0].astype(str), aux_pase.iloc[:, 1].astype(str)))

### Schema, options y correspondencias

In [43]:
EOD_STAGES_LVL2_SCHEMA = TripSchema(
    version="eod-stages-lvl2-sourceprofile",
    fields={
        "movement_id": FieldSpec("movement_id", "string", required=True),
        "trip_id": FieldSpec("trip_id", "string", required=True),
        "movement_seq": FieldSpec("movement_seq", "int", required=True),

        "origin_longitude": FieldSpec("origin_longitude", "float", required=True),
        "origin_latitude": FieldSpec("origin_latitude", "float", required=True),
        "destination_longitude": FieldSpec("destination_longitude", "float", required=True),
        "destination_latitude": FieldSpec("destination_latitude", "float", required=True),

        "origin_municipality": FieldSpec(
            "origin_municipality",
            "categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),
        "destination_municipality": FieldSpec(
            "destination_municipality",
            "categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),

        "mode": FieldSpec(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["car", "bus", "metro", "taxi", "walk", "bicycle", "other"],
                extendable=False,
            ),
        ),
        "payment_method": FieldSpec(
            "payment_method",
            "categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),

        # Campos desde Personas
        "user_gender": FieldSpec(
            "user_gender",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["male", "female", "unknown"],
                extendable=False,
            ),
        ),
        "Actividad_nombre": FieldSpec(
            "Actividad_nombre",
            "categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),
        "Ocupacion_nombre": FieldSpec(
            "Ocupacion_nombre",
            "categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),
        "LicenciaConducir_nombre": FieldSpec(
            "LicenciaConducir_nombre",
            "categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),
        "PaseEscolar_nombre": FieldSpec(
            "PaseEscolar_nombre",
            "categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),
    },
    required=[
        "movement_id",
        "trip_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
    ],
)

EOD_STAGES_LVL2_OPTIONS = ImportOptions(
    keep_extra_fields=False,
    selected_fields=None,
    strict=False,
    strict_domains=False,
    single_stage=True,
    source_timezone=None,
)

EOD_STAGES_LVL2_FIELD_CORRESPONDENCE = {
    "movement_id": "Etapa",
    "trip_id": "Viaje",
    "origin_municipality": "ComunaOrigen_nombre",
    "destination_municipality": "ComunaDestino_nombre",
    "mode": "Modo_nombre",
    "payment_method": "FormaPago_nombre",
    "user_gender": "Sexo_nombre",
}

EOD_STAGES_LVL2_VALUE_CORRESPONDENCE = {
    "mode": {
        "Auto Chofer": "car",
        "Auto Acompañante": "car",
        "Bus alimentador": "bus",
        "Bus troncal": "bus",
        "Bus urbano con pago al conductor (Metrobus y otros)": "bus",
        "Metro": "metro",
        "Taxi o radiotaxi": "taxi",
        "Taxi colectivo": "taxi",
        "Enteramente a pie": "walk",
        "Bicicleta": "bicycle",
        "Servicio Informal": "other",
        "Furgón escolar, como pasajero": "other",
    },
    "user_gender": {
        "Hombre": "male",
        "Mujer": "female",
    },
}

EOD_STAGES_LVL2_PROVENANCE = {
    "source": {
        "name": "EOD",
        "entity": "stages",
        "version": "EOD_STGO",
    },
    "notes": [
        "nivel 2 con SourceProfile",
        "solo 4-5 campos de etapa con tablas auxiliares",
        "5 campos desde Personas",
    ],
}

### Uso de SourceProfiles y preprocess

In [44]:
def preprocess_eod_stages_lvl2(df_etapas: pd.DataFrame) -> pd.DataFrame:
    # Trabajo sobre una copia para no tocar el dataframe original
    df = df_etapas.copy()

    # Traigo solo algunos campos de Personas
    person_cols = [
        "Hogar",
        "Persona",
        "Sexo",
        "Actividad",
        "Ocupacion",
        "LicenciaConducir",
        "PaseEscolar",
    ]

    df = df.merge(
        eod_personas[person_cols].copy(),
        on=["Hogar", "Persona"],
        how="left",
    )

    # Decodifico solo algunos campos de etapa
    df["ComunaOrigen_nombre"] = df["ComunaOrigen"].astype(str).map(comuna_map)
    df["ComunaDestino_nombre"] = df["ComunaDestino"].astype(str).map(comuna_map)
    df["Modo_nombre"] = df["Modo"].astype(str).map(modo_map)
    df["FormaPago_nombre"] = df["FormaPago"].astype(str).map(forma_pago_map)

    # Decodifico 5 campos desde Personas
    df["Sexo_nombre"] = df["Sexo"].astype(str).map(sexo_map)
    df["Actividad_nombre"] = df["Actividad"].astype(str).map(actividad_map)
    df["Ocupacion_nombre"] = df["Ocupacion"].astype(str).map(ocupacion_map)
    df["LicenciaConducir_nombre"] = df["LicenciaConducir"].astype(str).map(licencia_map)
    df["PaseEscolar_nombre"] = df["PaseEscolar"].astype(str).map(pase_map)

    # Conversión de coordenadas con el helper del módulo
    df = project_xy_to_latlon(
        df,
        x_col="OrigenCoordX",
        y_col="OrigenCoordY",
        source_crs="EPSG:5361",
        lon_col="origin_longitude",
        lat_col="origin_latitude",
        keep_debug_cols=False,
        drop_input_cols=False,
    )

    df = project_xy_to_latlon(
        df,
        x_col="DestinoCoordX",
        y_col="DestinoCoordY",
        source_crs="EPSG:5361",
        lon_col="destination_longitude",
        lat_col="destination_latitude",
        keep_debug_cols=False,
        drop_input_cols=False,
    )

    # Me quedo solo con lo que sí voy a usar
    cols_for_profile = [
        "Etapa",
        "Viaje",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "ComunaOrigen_nombre",
        "ComunaDestino_nombre",
        "Modo_nombre",
        "FormaPago_nombre",
        "Sexo_nombre",
        "Actividad_nombre",
        "Ocupacion_nombre",
        "LicenciaConducir_nombre",
        "PaseEscolar_nombre",
    ]

    return df[cols_for_profile].copy()


EOD_STAGES_LVL2_PROFILE = SourceProfile(
    name="EOD_STAGES_LVL2",
    description="Adaptación simplificada de EOD stages con SourceProfile",
    default_field_correspondence=EOD_STAGES_LVL2_FIELD_CORRESPONDENCE,
    default_value_correspondence=EOD_STAGES_LVL2_VALUE_CORRESPONDENCE,
    default_options=EOD_STAGES_LVL2_OPTIONS,
    preprocess=preprocess_eod_stages_lvl2,
    schema_override=EOD_STAGES_LVL2_SCHEMA,
)

In [40]:
df_test = preprocess_eod_stages_lvl2(eod_etapas.head(1000))
display(df_test.head())

,Etapa,Viaje,origin_longitude,origin_latitude,destination_longitude,destination_latitude,ComunaOrigen_nombre,ComunaDestino_nombre,Modo_nombre,FormaPago_nombre,Sexo_nombre,Actividad_nombre,Ocupacion_nombre,LicenciaConducir_nombre,PaseEscolar_nombre
0,10001001011,1000100101,-70.779034,-33.729444,-70.778858,-33.729996,BUIN,BUIN,Enteramente a pie,NaN,Hombre,Trabaja,Empleado u obrero del sector privado,"Licencia No Profesional Clase B: automoviles, camionetas, furgones, furgonetas, etc.",No
1,10001001021,1000100102,-70.778858,-33.729996,-70.779034,-33.729444,BUIN,BUIN,Enteramente a pie,NaN,Hombre,Trabaja,Empleado u obrero del sector privado,"Licencia No Profesional Clase B: automoviles, camionetas, furgones, furgonetas, etc.",No
2,10001002011,1000100201,-70.779034,-33.729444,-70.776804,-33.730774,BUIN,BUIN,Enteramente a pie,NaN,Mujer,Dueña(o) de casa,NaN,No tiene licencia de conducir,No
3,10001002021,1000100202,-70.776804,-33.730774,-70.776589,-33.730448,BUIN,BUIN,Enteramente a pie,NaN,Mujer,Dueña(o) de casa,NaN,No tiene licencia de conducir,No
4,10001002031,1000100203,-70.776589,-33.730448,-70.779034,-33.729444,BUIN,BUIN,Enteramente a pie,NaN,Mujer,Dueña(o) de casa,NaN,No tiene licencia de conducir,No


### Hacemos el import

In [45]:
# inspección clara del perfil
EOD_STAGES_LVL2_PROFILE.schema_override
EOD_STAGES_LVL2_PROFILE.default_options
EOD_STAGES_LVL2_PROFILE.default_field_correspondence
EOD_STAGES_LVL2_PROFILE.default_value_correspondence

trips_eod_stages_lvl2, report_eod_stages_lvl2 = import_trips_from_profile(
    profile=EOD_STAGES_LVL2_PROFILE,
    df=eod_etapas.head(1000),
    source_name="EOD stages - nivel 2 SourceProfile",
    provenance=EOD_STAGES_LVL2_PROVENANCE,
    h3_resolution=8,
)

display(trips_eod_stages_lvl2.data.head())
display(report_eod_stages_lvl2.issues[:10])

,movement_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_municipality,destination_municipality,mode,payment_method,user_gender,Actividad_nombre,Ocupacion_nombre,LicenciaConducir_nombre,PaseEscolar_nombre
0,10001001011,1000100101,0,-70.779034,-33.729444,-70.778858,-33.729996,BUIN,BUIN,walk,<NA>,male,Trabaja,Empleado u obrero del sector privado,"Licencia No Profesional Clase B: automoviles, camionetas, furgones, furgonetas, etc.",No
1,10001001021,1000100102,0,-70.778858,-33.729996,-70.779034,-33.729444,BUIN,BUIN,walk,<NA>,male,Trabaja,Empleado u obrero del sector privado,"Licencia No Profesional Clase B: automoviles, camionetas, furgones, furgonetas, etc.",No
2,10001002011,1000100201,0,-70.779034,-33.729444,-70.776804,-33.730774,BUIN,BUIN,walk,<NA>,female,Dueña(o) de casa,<NA>,No tiene licencia de conducir,No
3,10001002021,1000100202,0,-70.776804,-33.730774,-70.776589,-33.730448,BUIN,BUIN,walk,<NA>,female,Dueña(o) de casa,<NA>,No tiene licencia de conducir,No
4,10001002031,1000100203,0,-70.776589,-33.730448,-70.779034,-33.729444,BUIN,BUIN,walk,<NA>,female,Dueña(o) de casa,<NA>,No tiene licencia de conducir,No


[Issue(level='info', code='SCH.DOMAIN.EMPTY_VALUES', message="El DomainSpec del campo 'origin_municipality' tiene values vacío; se tratará como dominio bootstrap/extensible según política.", field='origin_municipality', source_field=None, row_count=None, details={'field': 'origin_municipality', 'values_size': 0, 'extendable': True, 'note': 'candidate_bootstrap'}),
 Issue(level='info', code='SCH.DOMAIN.EMPTY_VALUES', message="El DomainSpec del campo 'destination_municipality' tiene values vacío; se tratará como dominio bootstrap/extensible según política.", field='destination_municipality', source_field=None, row_count=None, details={'field': 'destination_municipality', 'values_size': 0, 'extendable': True, 'note': 'candidate_bootstrap'}),
 Issue(level='info', code='SCH.DOMAIN.EMPTY_VALUES', message="El DomainSpec del campo 'payment_method' tiene values vacío; se tratará como dominio bootstrap/extensible según política.", field='payment_method', source_field=None, row_count=None, detail

# Nivel 3

## Para EOD - Trips / viajes

### 1) Uso normal, rápido, sin decisiones de schema/campos/valores en la factory

In [8]:
import pandas as pd

from pylondrina.sources.helpers import import_trips_from_profile
from scripts.source_profiles.factories_eod.make_eod_trips_profile import (
    make_eod_trips_profile,
    EOD_TRIPS_DEFAULT_PROVENANCE_EXAMPLE,
)

eod_viajes = pd.read_csv(DATA_PATH / "Viajes.csv", sep=";", dtype=str, low_memory=False)
eod_personas = pd.read_csv(DATA_PATH / "Personas.csv", sep=";", dtype=str, low_memory=False)
eod_hogares = pd.read_csv(DATA_PATH / "Hogares.csv", sep=";", dtype=str, low_memory=False)

profile = make_eod_trips_profile(
    aux_dir=DATA_PATH / "Tablas_parametros",
    df_personas=eod_personas,
    df_hogares=eod_hogares,
    attach_person_fields=True,
    attach_household_fields=False,
)

# Convención de inspección clara
print("Inspeccion de trips:")
display(profile.schema_override)
display(profile.default_options)
display(profile.default_field_correspondence)
display(profile.default_value_correspondence)

Inspeccion de trips:


{'version': '1.1',
 'required': ['movement_id',
              'user_id',
              'origin_longitude',
              'origin_latitude',
              'destination_longitude',
              'destination_latitude',
              'origin_h3_index',
              'destination_h3_index',
              'trip_id',
              'movement_seq'],
 'semantic_rules': None,
 'fields': {'movement_id': {'name': 'movement_id',
                            'dtype': 'string',
                            'required': True,
                            'constraints': {'nullable': False, 'unique': True},
                            'domain': None},
            'user_id': {'name': 'user_id',
                        'dtype': 'string',
                        'required': True,
                        'constraints': {'nullable': False},
                        'domain': None},
            'origin_longitude': {'name': 'origin_longitude',
                                 'dtype': 'float',
                                 'required': True,
                                 'constraints': {'nullable': False, 'range': {'min': -180.0, 'max': 180.0}},
                                 'domain': None},
            'origin_latitude': {'name': 'origin_latitude',
                                'dtype': 'float',
                                'required': True,
                                'constraints': {'nullable': False, 'range': {'min': -90.0, 'max': 90.0}},
                                'domain': None},
            'destination_longitude': {'name': 'destination_longitude',
                                      'dtype': 'float',
                                      'required': True,
                                      'constraints': {'nullable': False, 'range': {'min': -180.0, 'max': 180.0}},
                                      'domain': None},
            'destination_latitude': {'name': 'destination_latitude',
                                     'dtype': 'float',
                                     'required': True,
                                     'constraints': {'nullable': False, 'range': {'min': -90.0, 'max': 90.0}},
                                     'domain': None},
            'origin_h3_index': {'name': 'origin_h3_index',
                                'dtype': 'string',
                                'required': True,
                                'constraints': {'nullable': False},
                                'domain': None},
            'destination_h3_index': {'name': 'destination_h3_index',
                                     'dtype': 'string',
                                     'required': True,
                                     'constraints': {'nullable': False},
                                     'domain': None},
            'trip_id': {'name': 'trip_id',
                        'dtype': 'string',
                        'required': True,
                        'constraints': {'nullable': False},
                        'domain': None},
            'movement_seq': {'name': 'movement_seq',
                             'dtype': 'int',
                             'required': True,
                             'constraints': {'nullable': False},
                             'domain': None},
            'origin_time_utc': {'name': 'origin_time_utc',
                                'dtype': 'datetime',
                                'required': False,
                                'constraints': None,
                                'domain': None},
            'destination_time_utc': {'name': 'destination_time_utc',
                                     'dtype': 'datetime',
                                     'required': False,
                                     'constraints': None,
                                     'domain': None},
            'origin_time_local_hhmm': {'name': 'origin_time_local_hhmm',
                                       'dtype': 'string',
                                       'r

ImportOptions(keep_extra_fields=True, selected_fields=None, strict=False, strict_domains=False, single_stage=True, source_timezone=None)

{'user_id': 'Persona',
 'movement_id': 'Viaje',
 'origin_longitude': 'OrigenCoordLon',
 'origin_latitude': 'OrigenCoordLat',
 'destination_longitude': 'DestinoCoordLon',
 'destination_latitude': 'DestinoCoordLat',
 'origin_time_local_hhmm': 'HoraIni',
 'destination_time_local_hhmm': 'HoraFin',
 'origin_municipality': 'ComunaOrigen',
 'destination_municipality': 'ComunaDestino',
 'mode': 'ModoAgregado',
 'purpose': 'Proposito',
 'day_type': 'TipoDia',
 'user_gender': 'Sexo',
 'trip_weight': 'factor_expansion'}

{'mode': {'Auto': 'car',
  'Bus TS': 'bus',
  'Bus no TS': 'bus',
  'Metro': 'metro',
  'Taxi Colectivo': 'taxi',
  'Taxi': 'taxi',
  'Caminata': 'walk',
  'Bicicleta': 'bicycle',
  'Bus TS - Bus no TS': 'bus',
  'Auto - Metro': 'other',
  'Bus TS - Metro': 'other',
  'Bus no TS - Metro': 'other',
  'Taxi Colectivo - Metro': 'other',
  'Taxi - Metro': 'other',
  'Otros - Metro': 'other',
  'Otros - Bus TS': 'other',
  'Otros - Bus TS - Metro': 'other',
  'Otros': 'other'},
 'purpose': {'volver a casa': 'home',
  'Volver a casa': 'home',
  'Al trabajo': 'work',
  'Por trabajo': 'work',
  'Al estudio': 'education',
  'Por estudio': 'education',
  'De compras': 'shopping',
  'Buscar o Dejar a alguien': 'errand',
  'Buscar o dejar algo': 'errand',
  'Trámites': 'errand',
  'De salud': 'health',
  'Comer o Tomar algo': 'leisure',
  'Visitar a alguien': 'leisure',
  'Recreación': 'leisure',
  'Otra actividad': 'other'},
 'day_type': {'Laboral': 'weekday', 'Fin de Semana': 'weekend'},
 'user_

In [9]:
trips_eod, report_eod = import_trips_from_profile(
    profile=profile,
    df=eod_viajes.head(5000),
    source_name="EOD trips level-3 factory",
    provenance=EOD_TRIPS_DEFAULT_PROVENANCE_EXAMPLE,
    h3_resolution=8,
)
display(trips_eod.data.head())
display(report_eod.issues)

,trip_id,Hogar,user_id,movement_id,Etapas,origin_municipality,destination_municipality,SectorOrigen,SectorDestino,ZonaOrigen,ZonaDestino,purpose,PropositoAgregado,ActividadDestino,MediosUsados,mode,ModoPriPub,ModoMotor,origin_time_local_hhmm,destination_time_local_hhmm,HoraMedia,TiempoViaje,TiempoMedio,Periodo,MinutosDespues,CuadrasDespues,FactorLaboralNormal,FactorSabadoNormal,FactorDomingoNormal,FactorLaboralEstival,FactorFindesemanaEstival,CodigoTiempo,AnoNac,user_gender,Actividad,Ocupacion,LicenciaConducir,PaseEscolar,AdultoMayor,TieneIngresos,TramoIngreso,mode_sequence,origin_longitude,origin_latitude,destination_longitude,destination_latitude,trip_weight,cantidad_etapas,origin_h3_index,destination_h3_index,movement_seq
0,1734310202,173431,17343102,1734310202,1,MAIPÚ,MAIPÚ,Poniente,Poniente,400,407,home,Otro,NaN,Bus alimentador,bus,Público,Motorizado,22:30,23:40,23:05,70,entre 1 hora y 1 hora 30 minutos,Noche (23:01 - 06:00),6,1,"1,00000000",NaN,NaN,NaN,NaN,Tiempo de viaje correcto,1972,female,Desempleado,NaN,No tiene licencia de conducir,No,No,no tiene ingresos,0,Bus alimentador,-70.774666,-33.531422,-70.735153,-33.495874,1.000000,1,88b2c540d1fffff,88b2c5429bfffff,0
1,1734410101,173441,17344101,1734410101,2,MAIPÚ,LAS CONDES,Poniente,Oriente,407,307,work,Trabajo,Servicios,Bus alimentador;Metro,other,Público,Motorizado,13:00,14:45,13:53,105,entre 1 hora 30 minutos y 2 horas,"Fuera de Punta 2 (9:01 - 10:00, 12:01 - 17:30, 20:31 - 23:00)",5,1,"1,12721985",NaN,NaN,NaN,NaN,Tiempo de viaje correcto,1991,male,Trabaja,Empleado u obrero del sector privado,No tiene licencia de conducir,No,No,"Sí, tiene ingresos",Entre 200.001 y 400.000 pesos,Bus alimentador;Metro,-70.738205,-33.500007,-70.567232,-33.408776,1.127220,2,88b2c5429bfffff,88b2c519a9fffff,0
2,1734410102,173441,17344101,1734410102,2,LAS CONDES,MAIPÚ,Oriente,Poniente,307,407,home,Trabajo,NaN,Metro;Bus alimentador,other,Público,Motorizado,22:00,23:30,22:45,90,entre 1 hora y 1 hora 30 minutos,"Fuera de Punta 2 (9:01 - 10:00, 12:01 - 17:30, 20:31 - 23:00)",10,2,"1,12721985",NaN,NaN,NaN,NaN,Tiempo de viaje correcto,1991,male,Trabaja,Empleado u obrero del sector privado,No tiene licencia de conducir,No,No,"Sí, tiene ingresos",Entre 200.001 y 400.000 pesos,Metro;Bus alimentador,-70.567232,-33.408776,-70.738205,-33.500007,1.127220,2,88b2c519a9fffff,88b2c5429bfffff,0
3,1734410301,173441,17344103,1734410301,2,MAIPÚ,ÑUÑOA,Poniente,Oriente,407,437,work,Trabajo,Servicios,Bus alimentador;Metro,other,Público,Motorizado,<NA>,<NA>,9:27,55,entre 31 minutos y 1 hora,"Fuera de Punta 2 (9:01 - 10:00, 12:01 - 17:30, 20:31 - 23:00)",10,2,"1,12721985",NaN,NaN,NaN,NaN,Tiempo de viaje correcto,1964,female,Trabaja,Empleado u obrero del sector privado,No tiene licencia de conducir,No,No,"Sí, tiene ingresos",No contesta,Bus alimentador;Metro,-70.738205,-33.500007,-70.604904,-33.454153,1.127220,2,88b2c5429bfffff,88b2c554dbfffff,0
4,1734410302,173441,17344103,1734410302,2,ÑUÑOA,MAIPÚ,Oriente,Poniente,437,407,home,Trabajo,NaN,Metro;Bus alimentador,other,Público,Motorizado,19:00,21:30,20:15,150,entre 2 horas y 2 horas 30 minutos,Punta Tarde (17:31 - 20:30),10,2,"1,05276356",NaN,NaN,NaN,NaN,Tiempo de viaje correcto,1964,female,Trabaja,Empleado u obrero del sector privado,No tiene licencia de conducir,No,No,"Sí, tiene ingresos",No contesta,Metro;Bus alimentador,-70.604904,-33.454153,-70.738205,-33.500007,1.052764,2,88b2c554dbfffff,88b2c5429bfffff,0


[Issue(level='info', code='IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND', message="El campo opcional 'day_type' no está presente en la fuente y no se encontró correspondencia; se omitirá.", field='day_type', source_field=None, row_count=None, details={'field': 'day_type', 'source_columns': ['Hogar', 'Persona', 'Viaje', 'Etapas', 'ComunaOrigen', 'ComunaDestino', 'SectorOrigen', 'SectorDestino', 'ZonaOrigen', 'ZonaDestino', 'Proposito', 'PropositoAgregado', 'ActividadDestino', 'MediosUsados', 'ModoAgregado', 'ModoPriPub', 'ModoMotor', 'HoraIni', 'HoraFin', 'HoraMedia', 'TiempoViaje', 'TiempoMedio', 'Periodo', 'MinutosDespues', 'CuadrasDespues', 'FactorLaboralNormal', 'FactorSabadoNormal', 'FactorDomingoNormal', 'FactorLaboralEstival', 'FactorFindesemanaEstival', 'CodigoTiempo', 'AnoNac', 'Sexo', 'Actividad', 'Ocupacion', 'LicenciaConducir', 'PaseEscolar', 'AdultoMayor', 'TieneIngresos', 'TramoIngreso', 'mode_sequence', 'OrigenCoordLon', 'OrigenCoordLat', 'DestinoCoordLon', 'DestinoCoordLat', 'fact

### 2) Uso más personalizado, con decisiones específicas

In [10]:
from pylondrina.sources.helpers import import_trips_from_profile
from scripts.source_profiles.factories_eod.make_eod_trips_profile import make_eod_trips_profile
from pylondrina.types import FieldCorrespondence, ValueCorrespondence
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec
from pylondrina.importing import ImportOptions

# -----------------------------------------------------------------------------
# Schema / options / correspondencias de ejemplo para uso más personalizado
# -----------------------------------------------------------------------------

EOD_TRIPS_CUSTOM_SCHEMA = TripSchema(
    version="1.1-eod-custom",
    fields={
        "movement_id": FieldSpec(
            "movement_id",
            "string",
            required=True,
            constraints={"nullable": False, "unique": True},
        ),
        "user_id": FieldSpec(
            "user_id",
            "string",
            required=True,
            constraints={"nullable": False},
        ),
        "origin_longitude": FieldSpec(
            "origin_longitude",
            "float",
            required=True,
            constraints={"nullable": False, "range": {"min": -180.0, "max": 180.0}},
        ),
        "origin_latitude": FieldSpec(
            "origin_latitude",
            "float",
            required=True,
            constraints={"nullable": False, "range": {"min": -90.0, "max": 90.0}},
        ),
        "destination_longitude": FieldSpec(
            "destination_longitude",
            "float",
            required=True,
            constraints={"nullable": False, "range": {"min": -180.0, "max": 180.0}},
        ),
        "destination_latitude": FieldSpec(
            "destination_latitude",
            "float",
            required=True,
            constraints={"nullable": False, "range": {"min": -90.0, "max": 90.0}},
        ),
        "origin_h3_index": FieldSpec(
            "origin_h3_index",
            "string",
            required=True,
            constraints={"nullable": False},
        ),
        "destination_h3_index": FieldSpec(
            "destination_h3_index",
            "string",
            required=True,
            constraints={"nullable": False},
        ),
        "trip_id": FieldSpec(
            "trip_id",
            "string",
            required=True,
            constraints={"nullable": False},
        ),
        "movement_seq": FieldSpec(
            "movement_seq",
            "int",
            required=True,
            constraints={"nullable": False},
        ),
        "origin_time_local_hhmm": FieldSpec("origin_time_local_hhmm", "string", required=False),
        "destination_time_local_hhmm": FieldSpec("destination_time_local_hhmm", "string", required=False),
        "origin_municipality": FieldSpec("origin_municipality", "string", required=False),
        "destination_municipality": FieldSpec("destination_municipality", "string", required=False),
        "mode": FieldSpec(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["walk", "bicycle", "car", "taxi", "bus", "metro", "train", "other"],
                extendable=True,
            ),
        ),
        "purpose": FieldSpec(
            "purpose",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["home", "work", "education", "shopping", "errand", "health", "leisure", "other"],
                extendable=True,
            ),
        ),
        "day_type": FieldSpec(
            "day_type",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["weekday", "weekend", "holiday"],
                extendable=True,
            ),
        ),
        "user_gender": FieldSpec(
            "user_gender",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["female", "male", "other", "unknown"],
                extendable=True,
            ),
        ),
        "trip_weight": FieldSpec("trip_weight", "float", required=False),
    },
    required=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "trip_id",
        "movement_seq",
    ],
)

EOD_TRIPS_CUSTOM_OPTIONS = ImportOptions(
    keep_extra_fields=False,
    selected_fields=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "trip_id",
        "movement_seq",
        "origin_time_local_hhmm",
        "destination_time_local_hhmm",
        "origin_municipality",
        "destination_municipality",
        "mode",
        "purpose",
        "day_type",
        "user_gender",
        "trip_weight",
    ],
    strict=False,
    strict_domains=False,
    single_stage=True,
    source_timezone=None,
)

# Ejemplo de override: usar propósito agregado en vez de propósito fino
EOD_TRIPS_CUSTOM_FIELD_CORRESPONDENCE: FieldCorrespondence = {
    "purpose": "PropositoAgregado",
}

EOD_TRIPS_CUSTOM_VALUE_CORRESPONDENCE: ValueCorrespondence = {
    "purpose": {
        "Trabajo y estudio": "other",
        "Otro" : "other",
        "Estudio": "education",
        "Trabajo": "work",
    },
    "mode": {
        "Tren": "train",
    },
}

EOD_TRIPS_CUSTOM_PROVENANCE_EXAMPLE = {
    "source": {
        "name": "EOD",
        "profile": "EOD_TRIPS_CUSTOM",
        "entity": "trips",
        "version": "EOD_STGO",
    },
    "notes": [
        "factory nivel 3 EOD con schema y correspondencias personalizadas",
        "purpose usa PropositoAgregado",
    ],
}

In [11]:
profile_custom = make_eod_trips_profile(
    aux_dir=DATA_PATH / "Tablas_parametros",
    df_personas=eod_personas,
    df_hogares=eod_hogares,
    attach_person_fields=True,
    attach_household_fields=True,
    schema_override=EOD_TRIPS_CUSTOM_SCHEMA,
    field_correspondence_override=EOD_TRIPS_CUSTOM_FIELD_CORRESPONDENCE,
    value_correspondence_override=EOD_TRIPS_CUSTOM_VALUE_CORRESPONDENCE,
    options_override={
        "keep_extra_fields": EOD_TRIPS_CUSTOM_OPTIONS.keep_extra_fields,
        "selected_fields": EOD_TRIPS_CUSTOM_OPTIONS.selected_fields,
        "strict": EOD_TRIPS_CUSTOM_OPTIONS.strict,
        "strict_domains": EOD_TRIPS_CUSTOM_OPTIONS.strict_domains,
        "single_stage": EOD_TRIPS_CUSTOM_OPTIONS.single_stage,
        "source_timezone": EOD_TRIPS_CUSTOM_OPTIONS.source_timezone,
    },
    profile_name="EOD_TRIPS_CUSTOM",
)

# Convención de inspección clara
print("Inspección:\n")
display(profile_custom.schema_override)
display(profile_custom.default_options)
display(profile_custom.default_field_correspondence)
display(profile_custom.default_value_correspondence)

Inspección:



{'version': '1.1-eod-custom',
 'required': ['movement_id',
              'user_id',
              'origin_longitude',
              'origin_latitude',
              'destination_longitude',
              'destination_latitude',
              'origin_h3_index',
              'destination_h3_index',
              'trip_id',
              'movement_seq'],
 'semantic_rules': None,
 'fields': {'movement_id': {'name': 'movement_id',
                            'dtype': 'string',
                            'required': True,
                            'constraints': {'nullable': False, 'unique': True},
                            'domain': None},
            'user_id': {'name': 'user_id',
                        'dtype': 'string',
                        'required': True,
                        'constraints': {'nullable': False},
                        'domain': None},
            'origin_longitude': {'name': 'origin_longitude',
                                 'dtype': 'float',
                                 'required': True,
                                 'constraints': {'nullable': False, 'range': {'min': -180.0, 'max': 180.0}},
                                 'domain': None},
            'origin_latitude': {'name': 'origin_latitude',
                                'dtype': 'float',
                                'required': True,
                                'constraints': {'nullable': False, 'range': {'min': -90.0, 'max': 90.0}},
                                'domain': None},
            'destination_longitude': {'name': 'destination_longitude',
                                      'dtype': 'float',
                                      'required': True,
                                      'constraints': {'nullable': False, 'range': {'min': -180.0, 'max': 180.0}},
                                      'domain': None},
            'destination_latitude': {'name': 'destination_latitude',
                                     'dtype': 'float',
                                     'required': True,
                                     'constraints': {'nullable': False, 'range': {'min': -90.0, 'max': 90.0}},
                                     'domain': None},
            'origin_h3_index': {'name': 'origin_h3_index',
                                'dtype': 'string',
                                'required': True,
                                'constraints': {'nullable': False},
                                'domain': None},
            'destination_h3_index': {'name': 'destination_h3_index',
                                     'dtype': 'string',
                                     'required': True,
                                     'constraints': {'nullable': False},
                                     'domain': None},
            'trip_id': {'name': 'trip_id',
                        'dtype': 'string',
                        'required': True,
                        'constraints': {'nullable': False},
                        'domain': None},
            'movement_seq': {'name': 'movement_seq',
                             'dtype': 'int',
                             'required': True,
                             'constraints': {'nullable': False},
                             'domain': None},
            'origin_time_local_hhmm': {'name': 'origin_time_local_hhmm',
                                       'dtype': 'string',
                                       'required': False,
                                       'constraints': None,
                                       'domain': None},
            'destination_time_local_hhmm': {'name': 'destination_time_local_hhmm',
                                            'dtype': 'string',
                                            'required': False,
                                            'constraints': None,
                                            'domain': None},
            'origin_municipality': {'name': 'origin_municipality',
              

ImportOptions(keep_extra_fields=False, selected_fields=['movement_id', 'user_id', 'origin_longitude', 'origin_latitude', 'destination_longitude', 'destination_latitude', 'origin_h3_index', 'destination_h3_index', 'trip_id', 'movement_seq', 'origin_time_local_hhmm', 'destination_time_local_hhmm', 'origin_municipality', 'destination_municipality', 'mode', 'purpose', 'day_type', 'user_gender', 'trip_weight'], strict=False, strict_domains=False, single_stage=True, source_timezone=None)

{'user_id': 'Persona',
 'movement_id': 'Viaje',
 'origin_longitude': 'OrigenCoordLon',
 'origin_latitude': 'OrigenCoordLat',
 'destination_longitude': 'DestinoCoordLon',
 'destination_latitude': 'DestinoCoordLat',
 'origin_time_local_hhmm': 'HoraIni',
 'destination_time_local_hhmm': 'HoraFin',
 'origin_municipality': 'ComunaOrigen',
 'destination_municipality': 'ComunaDestino',
 'mode': 'ModoAgregado',
 'purpose': 'PropositoAgregado',
 'day_type': 'TipoDia',
 'user_gender': 'Sexo',
 'trip_weight': 'factor_expansion'}

{'mode': {'Auto': 'car',
  'Bus TS': 'bus',
  'Bus no TS': 'bus',
  'Metro': 'metro',
  'Taxi Colectivo': 'taxi',
  'Taxi': 'taxi',
  'Caminata': 'walk',
  'Bicicleta': 'bicycle',
  'Bus TS - Bus no TS': 'bus',
  'Auto - Metro': 'other',
  'Bus TS - Metro': 'other',
  'Bus no TS - Metro': 'other',
  'Taxi Colectivo - Metro': 'other',
  'Taxi - Metro': 'other',
  'Otros - Metro': 'other',
  'Otros - Bus TS': 'other',
  'Otros - Bus TS - Metro': 'other',
  'Otros': 'other',
  'Tren': 'train'},
 'purpose': {'volver a casa': 'home',
  'Volver a casa': 'home',
  'Al trabajo': 'work',
  'Por trabajo': 'work',
  'Al estudio': 'education',
  'Por estudio': 'education',
  'De compras': 'shopping',
  'Buscar o Dejar a alguien': 'errand',
  'Buscar o dejar algo': 'errand',
  'Trámites': 'errand',
  'De salud': 'health',
  'Comer o Tomar algo': 'leisure',
  'Visitar a alguien': 'leisure',
  'Recreación': 'leisure',
  'Otra actividad': 'other',
  'Trabajo y estudio': 'other',
  'Otro': 'other',
  '

In [12]:

trips_eod_custom, report_eod_custom = import_trips_from_profile(
    profile=profile_custom,
    df=eod_viajes.head(5000),
    source_name="EOD trips level-3 custom factory",
    provenance=EOD_TRIPS_CUSTOM_PROVENANCE_EXAMPLE,
    h3_resolution=8,
)


In [13]:
display(trips_eod_custom.data[1:20])
display(report_eod_custom.issues)

,trip_id,user_id,movement_id,origin_municipality,destination_municipality,purpose,mode,origin_time_local_hhmm,destination_time_local_hhmm,user_gender,day_type,origin_longitude,origin_latitude,destination_longitude,destination_latitude,trip_weight,origin_h3_index,destination_h3_index,movement_seq
1,1734410101,17344101,1734410101,MAIPÚ,LAS CONDES,work,other,13:00,14:45,male,weekday,-70.738205,-33.500007,-70.567232,-33.408776,1.127220,88b2c5429bfffff,88b2c519a9fffff,0
2,1734410102,17344101,1734410102,LAS CONDES,MAIPÚ,work,other,22:00,23:30,male,weekday,-70.567232,-33.408776,-70.738205,-33.500007,1.127220,88b2c519a9fffff,88b2c5429bfffff,0
3,1734410301,17344103,1734410301,MAIPÚ,ÑUÑOA,work,other,<NA>,<NA>,female,weekday,-70.738205,-33.500007,-70.604904,-33.454153,1.127220,88b2c5429bfffff,88b2c554dbfffff,0
4,1734410302,17344103,1734410302,ÑUÑOA,MAIPÚ,work,other,19:00,21:30,female,weekday,-70.604904,-33.454153,-70.738205,-33.500007,1.052764,88b2c554dbfffff,88b2c5429bfffff,0
5,1734510101,17345101,1734510101,MAIPÚ,SANTIAGO,other,car,10:00,11:00,female,weekend,-70.786138,-33.556823,-70.655200,-33.440072,1.482104,88b2c540bbfffff,88b2c55411fffff,0
6,1734510102,17345101,1734510102,SANTIAGO,MAIPÚ,other,car,15:00,15:45,female,weekend,-70.655200,-33.440072,-70.786138,-33.556823,1.482104,88b2c55411fffff,88b2c540bbfffff,0
7,1734510103,17345101,1734510103,MAIPÚ,MAIPÚ,home,walk,17:00,17:20,female,weekend,NaN,NaN,NaN,NaN,1.482104,<NA>,<NA>,0
8,1734510104,17345101,1734510104,MAIPÚ,MAIPÚ,home,walk,18:00,18:20,female,weekend,NaN,NaN,NaN,NaN,1.482104,<NA>,<NA>,0
9,1734510201,17345102,1734510201,MAIPÚ,MAIPÚ,other,walk,13:00,13:20,male,weekend,NaN,NaN,NaN,NaN,1.482104,<NA>,<NA>,0
10,1734510202,17345102,1734510202,MAIPÚ,MAIPÚ,other,walk,14:20,14:45,male,weekend,NaN,NaN,NaN,NaN,1.482104,<NA>,<NA>,0


[Issue(level='warning', code='IMP.TEMPORAL.TIER_LIMITED', message='El dataset fue importado con capacidad temporal limitada (tier_2); algunas operaciones posteriores quedarán restringidas.', field=None, source_field=None, row_count=None, details={'temporal_tier': 'tier_2', 'fields_present': ['origin_time_local_hhmm', 'destination_time_local_hhmm'], 'note': 'limited_temporal_capabilities'}),
 Issue(level='warning', code='IMP.TYPE.COERCE_PARTIAL', message="La conversión mínima del campo 'origin_time_local_hhmm' falló en algunas filas (26.6%); se marcarán como nulos para validación posterior.", field='origin_time_local_hhmm', source_field=None, row_count=None, details={'field': 'origin_time_local_hhmm', 'dtype_expected': 'string_hhmm', 'parse_fail_count': 1331, 'total_count': 5000, 'fail_rate': 0.2662, 'fallback': 'set_null', 'action': 'continue'}),
 Issue(level='warning', code='IMP.TYPE.COERCE_PARTIAL', message="La conversión mínima del campo 'origin_time_local_hhmm' falló en algunas fil

## Para EOD - Stages / etapas

### 1) Uso normal, rápido, sin decisiones de schema/campos/valores en la factory

In [14]:
import pandas as pd

from pylondrina.sources.helpers import import_trips_from_profile
from scripts.source_profiles.factories_eod.make_eod_stages_profile import make_eod_stages_profile
from scripts.source_profiles.factories_eod.stages_defaults import (
    EOD_STAGES_DEFAULT_PROVENANCE_EXAMPLE,
)

eod_etapas = pd.read_csv(DATA_PATH / "Etapas.csv", sep=";", dtype=str, low_memory=False, encoding="latin-1")
eod_viajes = pd.read_csv(DATA_PATH / "Viajes.csv", sep=";", dtype=str, low_memory=False)
eod_personas = pd.read_csv(DATA_PATH / "Personas.csv", sep=";", dtype=str, low_memory=False)
eod_hogares = pd.read_csv(DATA_PATH / "Hogares.csv", sep=";", dtype=str, low_memory=False)

In [15]:
profile = make_eod_stages_profile(
    aux_dir=DATA_PATH / "Tablas_parametros",
    df_viajes=eod_viajes,
    df_personas=eod_personas,
    df_hogares=eod_hogares,
    attach_trip_context=True,
    attach_person_fields=True,
    attach_household_fields=False,
)

In [16]:
# Inspección clara
print("Inspeccion: ")
display(profile.schema_override)
display(profile.default_options)
display(profile.default_field_correspondence)
display(profile.default_value_correspondence)

Inspeccion: 


{'version': '1.1',
 'required': ['movement_id',
              'user_id',
              'origin_longitude',
              'origin_latitude',
              'destination_longitude',
              'destination_latitude',
              'origin_h3_index',
              'destination_h3_index',
              'trip_id',
              'movement_seq'],
 'semantic_rules': None,
 'fields': {'movement_id': {'name': 'movement_id',
                            'dtype': 'string',
                            'required': True,
                            'constraints': {'nullable': False, 'unique': True},
                            'domain': None},
            'user_id': {'name': 'user_id',
                        'dtype': 'string',
                        'required': True,
                        'constraints': {'nullable': False},
                        'domain': None},
            'origin_longitude': {'name': 'origin_longitude',
                                 'dtype': 'float',
                                 'required': True,
                                 'constraints': {'nullable': False, 'range': {'min': -180.0, 'max': 180.0}},
                                 'domain': None},
            'origin_latitude': {'name': 'origin_latitude',
                                'dtype': 'float',
                                'required': True,
                                'constraints': {'nullable': False, 'range': {'min': -90.0, 'max': 90.0}},
                                'domain': None},
            'destination_longitude': {'name': 'destination_longitude',
                                      'dtype': 'float',
                                      'required': True,
                                      'constraints': {'nullable': False, 'range': {'min': -180.0, 'max': 180.0}},
                                      'domain': None},
            'destination_latitude': {'name': 'destination_latitude',
                                     'dtype': 'float',
                                     'required': True,
                                     'constraints': {'nullable': False, 'range': {'min': -90.0, 'max': 90.0}},
                                     'domain': None},
            'origin_h3_index': {'name': 'origin_h3_index',
                                'dtype': 'string',
                                'required': True,
                                'constraints': {'nullable': False},
                                'domain': None},
            'destination_h3_index': {'name': 'destination_h3_index',
                                     'dtype': 'string',
                                     'required': True,
                                     'constraints': {'nullable': False},
                                     'domain': None},
            'trip_id': {'name': 'trip_id',
                        'dtype': 'string',
                        'required': True,
                        'constraints': {'nullable': False},
                        'domain': None},
            'movement_seq': {'name': 'movement_seq',
                             'dtype': 'int',
                             'required': True,
                             'constraints': {'nullable': False, 'range': {'min': 1}},
                             'domain': None},
            'origin_time_utc': {'name': 'origin_time_utc',
                                'dtype': 'datetime',
                                'required': False,
                                'constraints': None,
                                'domain': None},
            'destination_time_utc': {'name': 'destination_time_utc',
                                     'dtype': 'datetime',
                                     'required': False,
                                     'constraints': None,
                                     'domain': None},
            'origin_time_local_hhmm': {'name': 'origin_time_local_hhmm',
                                       'dtype': 'string',
                    

ImportOptions(keep_extra_fields=True, selected_fields=None, strict=False, strict_domains=False, single_stage=False, source_timezone=None)

{'user_id': 'Persona',
 'trip_id': 'Viaje',
 'movement_id': 'movement_id_src',
 'movement_seq': 'NumeroEtapa',
 'origin_longitude': 'OrigenCoordLon',
 'origin_latitude': 'OrigenCoordLat',
 'destination_longitude': 'DestinoCoordLon',
 'destination_latitude': 'DestinoCoordLat',
 'origin_municipality': 'ComunaOrigen',
 'destination_municipality': 'ComunaDestino',
 'mode': 'Modo',
 'purpose': 'trip_Proposito',
 'day_type': 'TipoDia',
 'user_gender': 'Sexo',
 'trip_weight': 'factor_expansion'}

{'mode': {'Auto Chofer': 'car',
  'Auto Acompañante': 'car',
  'Bus alimentador': 'bus',
  'Bus troncal': 'bus',
  'Bus institucional': 'bus',
  'Bus interurbano o rural': 'bus',
  'Bus urbano con pago al conductor (Metrobus y otros)': 'bus',
  'Metro': 'metro',
  'Taxi colectivo': 'taxi',
  'Taxi o radiotaxi': 'taxi',
  'Enteramente a pie': 'walk',
  'Bicicleta': 'bicycle',
  'Motocicleta': 'motorcycle',
  'Motocicleta Acompañante': 'motorcycle',
  'Tren': 'train',
  'Servicio Informal': 'other',
  'Furgón escolar, como pasajero': 'other',
  'Furgón escolar, como chofer o acompañante': 'other'},
 'purpose': {'volver a casa': 'home',
  'Volver a casa': 'home',
  'Al trabajo': 'work',
  'Por trabajo': 'work',
  'Al estudio': 'education',
  'Por estudio': 'education',
  'De compras': 'shopping',
  'Buscar o Dejar a alguien': 'errand',
  'Buscar o dejar algo': 'errand',
  'Trámites': 'errand',
  'De salud': 'health',
  'Comer o Tomar algo': 'leisure',
  'Visitar a alguien': 'leisure',
  '

In [17]:
trips_eod_stages, report_eod_stages = import_trips_from_profile(
    profile=profile,
    df=eod_etapas.head(5000),
    source_name="EOD stages level-3 factory",
    provenance=EOD_STAGES_DEFAULT_PROVENANCE_EXAMPLE,
    h3_resolution=8,
)
display(trips_eod_stages.data.head())
display(report_eod_stages.issues)

,Hogar,user_id,trip_id,Etapa,ZonaOrigen,ZonaDestino,origin_municipality,destination_municipality,mode,CuadrasAntes,MinutosAntes,Autopistas,NoUsaAutopistas,Estaciona,CostoEstacionamiento,FormaPago,EstacionTrenIni,EstacionTrenFin,TarifaTren,RecorridoTransantiago,TiempoEsperaTstgo,TiempoEsperaBus,BusesPerdidos,TarifaBusNoTransantiago,EstacionMetroIni,EstacionMetroFin,HorarioMetro,MetrosPerdidos,EstacionMetroCambio,RecorridoTxc,TiempoEsperaTxc,TarifaTxc,TiempoEsperaTaxi,TarifaTaxi,PropiedadBicicleta,UsaCiclovia,CirculacionBicicleta,EstacionaBicicleta,ModoEstacionaBicicleta,UsoHabitualBicicleta,AnoNac,user_gender,Actividad,Ocupacion,LicenciaConducir,PaseEscolar,AdultoMayor,TieneIngresos,TramoIngreso,trip_Etapas,purpose,trip_PropositoAgregado,trip_ActividadDestino,trip_ModoAgregado,trip_HoraIni,trip_HoraFin,trip_TiempoViaje,trip_TiempoMedio,trip_Periodo,trip_FactorLaboralNormal,trip_FactorSabadoNormal,trip_FactorDomingoNormal,trip_FactorLaboralEstival,trip_FactorFindesemanaEstival,origin_longitude,origin_latitude,destination_longitude,destination_latitude,movement_seq,movement_id,trip_weight,cantidad_etapas,origin_h3_index,destination_h3_index
0,100010,10001001,1000100101,10001001011,786,786,BUIN,BUIN,walk,0,0,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1958,male,Trabaja,Empleado u obrero del sector privado,"Licencia No Profesional Clase B: automoviles, camionetas, furgones, furgonetas, etc.",No,No,"Sí, tiene ingresos",Entre 200.001 y 400.000 pesos,1,work,Trabajo,"Habitacional (Ej. Servicio doméstico, enfermera a domicilio)",Caminata,15:00,15:02,2,menor o igual a 30 minutos,"Fuera de Punta 2 (9:01 - 10:00, 12:01 - 17:30, 20:31 - 23:00)",NaN,NaN,"1,48210391",NaN,NaN,-70.779034,-33.729444,-70.778858,-33.729996,1,10001001011,1.482104,1,88b2c5792bfffff,88b2c5793dfffff
1,100010,10001001,1000100102,10001001021,786,786,BUIN,BUIN,walk,0,0,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1958,male,Trabaja,Empleado u obrero del sector privado,"Licencia No Profesional Clase B: automoviles, camionetas, furgones, furgonetas, etc.",No,No,"Sí, tiene ingresos",Entre 200.001 y 400.000 pesos,1,home,Trabajo,NaN,Caminata,20:00,20:02,2,menor o igual a 30 minutos,Punta Tarde (17:31 - 20:30),NaN,NaN,"1,48210391",NaN,NaN,-70.778858,-33.729996,-70.779034,-33.729444,1,10001001021,1.482104,1,88b2c5793dfffff,88b2c5792bfffff
2,100010,10001002,1000100201,10001002011,786,786,BUIN,BUIN,walk,0,0,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1956,female,Dueña(o) de casa,NaN,No tiene licencia de conducir,No,No,no tiene ingresos,0,1,shopping,Otro,NaN,Caminata,11:30,11:36,6,menor o igual a 30 minutos,Fuera de Punta 1 (10:01 - 12:00),NaN,NaN,"1,48210391",NaN,NaN,-70.779034,-33.729444,-70.776804,-33.730774,1,10001002011,1.482104,1,88b2c5792bfffff,88b2c5793dfffff
3,100010,10001002,1000100202,10001002021,786,786,BUIN,BUIN,walk,0,0,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1956,female,Dueña(o) de casa,NaN,No tiene licencia de conducir,No,No,no tiene ingresos,0,1,shopping,Otro,NaN,Caminata,12:06,12:07,1,menor o igual a 30 minutos,"Fuera de Punta 2 (9:01 - 10:00, 12:01 - 17:30, 20:31 - 23:00)",NaN,NaN,"1,48210391",NaN,NaN,-70.776804,-33.730774,-70.776589,-33.730448,1,10001002021,1.482104,1,88b2c5793dfffff,88b2c5793dfffff
4,100010,10001002,1000100203,10001002031,786,786,BUIN,BUIN,walk,0,0,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1956,female,Dueña(o) de casa,NaN,No tiene licencia de conducir,No,No,no tiene ingresos,0,1,home,Otro,NaN,Caminata,12:13,12:17,4,menor o igual a 30 minutos,"Fuera de Punta 2 (9:01 - 10:00, 12:01 - 17:30, 20:31 - 23:00)",NaN,NaN,"1,48210391",NaN,NaN,-70.776589,-33.730448,-70.779034,-33.729444,1,10001002031,1

[Issue(level='info', code='IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND', message="El campo opcional 'day_type' no está presente en la fuente y no se encontró correspondencia; se omitirá.", field='day_type', source_field=None, row_count=None, details={'field': 'day_type', 'source_columns': ['Hogar', 'Persona', 'Viaje', 'Etapa', 'ZonaOrigen', 'ZonaDestino', 'ComunaOrigen', 'ComunaDestino', 'Modo', 'CuadrasAntes', 'MinutosAntes', 'Autopistas', 'NoUsaAutopistas', 'Estaciona', 'CostoEstacionamiento', 'FormaPago', 'EstacionTrenIni', 'EstacionTrenFin', 'TarifaTren', 'RecorridoTransantiago', 'TiempoEsperaTstgo', 'TiempoEsperaBus', 'BusesPerdidos', 'TarifaBusNoTransantiago', 'EstacionMetroIni', 'EstacionMetroFin', 'HorarioMetro', 'MetrosPerdidos', 'EstacionMetroCambio', 'RecorridoTxc', 'TiempoEsperaTxc', 'TarifaTxc', 'TiempoEsperaTaxi', 'TarifaTaxi', 'PropiedadBicicleta', 'UsaCiclovia', 'CirculacionBicicleta', 'EstacionaBicicleta', 'ModoEstacionaBicicleta', 'UsoHabitualBicicleta', 'AnoNac', 'Sexo', 'Act

### 2) Uso más personalizado, con decisiones específicas

In [18]:
from pylondrina.sources.helpers import import_trips_from_profile
from scripts.source_profiles.factories_eod.make_eod_trips_profile import make_eod_trips_profile
from pylondrina.types import FieldCorrespondence, ValueCorrespondence
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec
from pylondrina.importing import ImportOptions

# -----------------------------------------------------------------------------
# Objetos de ejemplo para uso más personalizado
# -----------------------------------------------------------------------------

EOD_STAGES_CUSTOM_SCHEMA = TripSchema(
    version="1.1-eod-stages-custom",
    fields={
        "movement_id": FieldSpec(
            "movement_id",
            "string",
            required=True,
            constraints={"nullable": False, "unique": True},
        ),
        "user_id": FieldSpec(
            "user_id",
            "string",
            required=True,
            constraints={"nullable": False},
        ),
        "trip_id": FieldSpec(
            "trip_id",
            "string",
            required=True,
            constraints={"nullable": False},
        ),
        "movement_seq": FieldSpec(
            "movement_seq",
            "int",
            required=True,
            constraints={"nullable": False, "range": {"min": 1}},
        ),
        "origin_longitude": FieldSpec(
            "origin_longitude",
            "float",
            required=True,
            constraints={"nullable": False},
        ),
        "origin_latitude": FieldSpec(
            "origin_latitude",
            "float",
            required=True,
            constraints={"nullable": False},
        ),
        "destination_longitude": FieldSpec(
            "destination_longitude",
            "float",
            required=True,
            constraints={"nullable": False},
        ),
        "destination_latitude": FieldSpec(
            "destination_latitude",
            "float",
            required=True,
            constraints={"nullable": False},
        ),
        "origin_h3_index": FieldSpec(
            "origin_h3_index",
            "string",
            required=True,
            constraints={"nullable": False},
        ),
        "destination_h3_index": FieldSpec(
            "destination_h3_index",
            "string",
            required=True,
            constraints={"nullable": False},
        ),
        "origin_municipality": FieldSpec("origin_municipality", "string", required=False),
        "destination_municipality": FieldSpec("destination_municipality", "string", required=False),
        "mode": FieldSpec(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["walk", "bicycle", "motorcycle", "car", "taxi", "bus", "metro", "train", "other"],
                extendable=True,
            ),
        ),
        "purpose": FieldSpec(
            "purpose",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["home", "work", "education", "shopping", "errand", "health", "leisure", "other"],
                extendable=True,
            ),
        ),
        "day_type": FieldSpec(
            "day_type",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["weekday", "weekend", "holiday"],
                extendable=True,
            ),
        ),
        "user_gender": FieldSpec(
            "user_gender",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["female", "male", "other", "unknown"],
                extendable=True,
            ),
        ),
        "trip_weight": FieldSpec("trip_weight", "float", required=False),
    },
    required=[
        "movement_id",
        "user_id",
        "trip_id",
        "movement_seq",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
    ],
)

EOD_STAGES_CUSTOM_OPTIONS = ImportOptions(
    keep_extra_fields=False,
    selected_fields=[
        "movement_id",
        "user_id",
        "trip_id",
        "movement_seq",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "origin_municipality",
        "destination_municipality",
        "mode",
        "purpose",
        "day_type",
        "user_gender",
        "trip_weight",
    ],
    strict=False,
    strict_domains=False,
    single_stage=False,
    source_timezone=None,
)

EOD_STAGES_CUSTOM_FIELD_CORRESPONDENCE: FieldCorrespondence = {
    "purpose": "trip_PropositoAgregado",
}

EOD_STAGES_CUSTOM_VALUE_CORRESPONDENCE: ValueCorrespondence = {
    "purpose": {
        "Trabajo y estudio": "other",
    },
    "mode": {
        "Metrotren Nos": "train",
    },
}

EOD_STAGES_CUSTOM_PROVENANCE_EXAMPLE = {
    "source": {
        "name": "EOD",
        "profile": "EOD_STAGES_CUSTOM",
        "entity": "stages",
        "version": "EOD_STGO",
    },
    "notes": [
        "factory nivel 3 EOD stages con schema y correspondencias personalizadas",
        "purpose usa trip_PropositoAgregado",
    ],
}

In [19]:

profile_custom = make_eod_stages_profile(
    aux_dir=DATA_PATH / "Tablas_parametros",
    df_viajes=eod_viajes,
    df_personas=eod_personas,
    df_hogares=eod_hogares,
    attach_trip_context=True,
    attach_person_fields=True,
    attach_household_fields=True,
    schema_override=EOD_STAGES_CUSTOM_SCHEMA,
    field_correspondence_override=EOD_STAGES_CUSTOM_FIELD_CORRESPONDENCE,
    value_correspondence_override=EOD_STAGES_CUSTOM_VALUE_CORRESPONDENCE,
    options_override={
        "keep_extra_fields": EOD_STAGES_CUSTOM_OPTIONS.keep_extra_fields,
        "selected_fields": EOD_STAGES_CUSTOM_OPTIONS.selected_fields,
        "strict": EOD_STAGES_CUSTOM_OPTIONS.strict,
        "strict_domains": EOD_STAGES_CUSTOM_OPTIONS.strict_domains,
        "single_stage": EOD_STAGES_CUSTOM_OPTIONS.single_stage,
        "source_timezone": EOD_STAGES_CUSTOM_OPTIONS.source_timezone,
    },
    profile_name="EOD_STAGES_CUSTOM",
)

# Inspección clara
print("Inspeccion")
display(profile_custom.schema_override)
display(profile_custom.default_options)
display(profile_custom.default_field_correspondence)
display(profile_custom.default_value_correspondence)



Inspeccion


{'version': '1.1-eod-stages-custom',
 'required': ['movement_id',
              'user_id',
              'trip_id',
              'movement_seq',
              'origin_longitude',
              'origin_latitude',
              'destination_longitude',
              'destination_latitude',
              'origin_h3_index',
              'destination_h3_index'],
 'semantic_rules': None,
 'fields': {'movement_id': {'name': 'movement_id',
                            'dtype': 'string',
                            'required': True,
                            'constraints': {'nullable': False, 'unique': True},
                            'domain': None},
            'user_id': {'name': 'user_id',
                        'dtype': 'string',
                        'required': True,
                        'constraints': {'nullable': False},
                        'domain': None},
            'trip_id': {'name': 'trip_id',
                        'dtype': 'string',
                        'required': True,
                        'constraints': {'nullable': False},
                        'domain': None},
            'movement_seq': {'name': 'movement_seq',
                             'dtype': 'int',
                             'required': True,
                             'constraints': {'nullable': False, 'range': {'min': 1}},
                             'domain': None},
            'origin_longitude': {'name': 'origin_longitude',
                                 'dtype': 'float',
                                 'required': True,
                                 'constraints': {'nullable': False},
                                 'domain': None},
            'origin_latitude': {'name': 'origin_latitude',
                                'dtype': 'float',
                                'required': True,
                                'constraints': {'nullable': False},
                                'domain': None},
            'destination_longitude': {'name': 'destination_longitude',
                                      'dtype': 'float',
                                      'required': True,
                                      'constraints': {'nullable': False},
                                      'domain': None},
            'destination_latitude': {'name': 'destination_latitude',
                                     'dtype': 'float',
                                     'required': True,
                                     'constraints': {'nullable': False},
                                     'domain': None},
            'origin_h3_index': {'name': 'origin_h3_index',
                                'dtype': 'string',
                                'required': True,
                                'constraints': {'nullable': False},
                                'domain': None},
            'destination_h3_index': {'name': 'destination_h3_index',
                                     'dtype': 'string',
                                     'required': True,
                                     'constraints': {'nullable': False},
                                     'domain': None},
            'origin_municipality': {'name': 'origin_municipality',
                                    'dtype': 'string',
                                    'required': False,
                                    'constraints': None,
                                    'domain': None},
            'destination_municipality': {'name': 'destination_municipality',
                                         'dtype': 'string',
                                         'required': False,
                                         'constraints': None,
                                         'domain': None},
            'mode': {'name': 'mode',
                     'dtype': 'categorical',
                     'required': False,
                     'constraints': None,
                     'domain': {'values': ['walk',
                              

ImportOptions(keep_extra_fields=False, selected_fields=['movement_id', 'user_id', 'trip_id', 'movement_seq', 'origin_longitude', 'origin_latitude', 'destination_longitude', 'destination_latitude', 'origin_h3_index', 'destination_h3_index', 'origin_municipality', 'destination_municipality', 'mode', 'purpose', 'day_type', 'user_gender', 'trip_weight'], strict=False, strict_domains=False, single_stage=False, source_timezone=None)

{'user_id': 'Persona',
 'trip_id': 'Viaje',
 'movement_id': 'movement_id_src',
 'movement_seq': 'NumeroEtapa',
 'origin_longitude': 'OrigenCoordLon',
 'origin_latitude': 'OrigenCoordLat',
 'destination_longitude': 'DestinoCoordLon',
 'destination_latitude': 'DestinoCoordLat',
 'origin_municipality': 'ComunaOrigen',
 'destination_municipality': 'ComunaDestino',
 'mode': 'Modo',
 'purpose': 'trip_PropositoAgregado',
 'day_type': 'TipoDia',
 'user_gender': 'Sexo',
 'trip_weight': 'factor_expansion'}

{'mode': {'Auto Chofer': 'car',
  'Auto Acompañante': 'car',
  'Bus alimentador': 'bus',
  'Bus troncal': 'bus',
  'Bus institucional': 'bus',
  'Bus interurbano o rural': 'bus',
  'Bus urbano con pago al conductor (Metrobus y otros)': 'bus',
  'Metro': 'metro',
  'Taxi colectivo': 'taxi',
  'Taxi o radiotaxi': 'taxi',
  'Enteramente a pie': 'walk',
  'Bicicleta': 'bicycle',
  'Motocicleta': 'motorcycle',
  'Motocicleta Acompañante': 'motorcycle',
  'Tren': 'train',
  'Servicio Informal': 'other',
  'Furgón escolar, como pasajero': 'other',
  'Furgón escolar, como chofer o acompañante': 'other',
  'Metrotren Nos': 'train'},
 'purpose': {'volver a casa': 'home',
  'Volver a casa': 'home',
  'Al trabajo': 'work',
  'Por trabajo': 'work',
  'Al estudio': 'education',
  'Por estudio': 'education',
  'De compras': 'shopping',
  'Buscar o Dejar a alguien': 'errand',
  'Buscar o dejar algo': 'errand',
  'Trámites': 'errand',
  'De salud': 'health',
  'Comer o Tomar algo': 'leisure',
  'Visita

In [20]:
trips_eod_stages_custom, report_eod_stages_custom = import_trips_from_profile(
    profile=profile_custom,
    df=eod_etapas.head(5000),
    source_name="EOD stages level-3 custom factory",
    provenance=EOD_STAGES_CUSTOM_PROVENANCE_EXAMPLE,
    h3_resolution=8,
)
display(trips_eod_stages_custom.data.head())
display(report_eod_stages_custom.issues)

,user_id,trip_id,origin_municipality,destination_municipality,mode,user_gender,day_type,purpose,origin_longitude,origin_latitude,destination_longitude,destination_latitude,movement_seq,movement_id,trip_weight,origin_h3_index,destination_h3_index
0,10001001,1000100101,BUIN,BUIN,walk,male,weekend,Trabajo,-70.779034,-33.729444,-70.778858,-33.729996,1,10001001011,1.482104,88b2c5792bfffff,88b2c5793dfffff
1,10001001,1000100102,BUIN,BUIN,walk,male,weekend,Trabajo,-70.778858,-33.729996,-70.779034,-33.729444,1,10001001021,1.482104,88b2c5793dfffff,88b2c5792bfffff
2,10001002,1000100201,BUIN,BUIN,walk,female,weekend,Otro,-70.779034,-33.729444,-70.776804,-33.730774,1,10001002011,1.482104,88b2c5792bfffff,88b2c5793dfffff
3,10001002,1000100202,BUIN,BUIN,walk,female,weekend,Otro,-70.776804,-33.730774,-70.776589,-33.730448,1,10001002021,1.482104,88b2c5793dfffff,88b2c5793dfffff
4,10001002,1000100203,BUIN,BUIN,walk,female,weekend,Otro,-70.776589,-33.730448,-70.779034,-33.729444,1,10001002031,1.482104,88b2c5793dfffff,88b2c5792bfffff


[Issue(level='warning', code='IMP.TEMPORAL.TIER_LIMITED', message='El dataset fue importado con capacidad temporal limitada (tier_3); algunas operaciones posteriores quedarán restringidas.', field=None, source_field=None, row_count=None, details={'temporal_tier': 'tier_3', 'fields_present': [], 'note': 'limited_temporal_capabilities'}),
 Issue(level='info', code='DOM.EXTENSION.APPLIED', message="Se extendió el dominio de 'purpose' con 3 valores nuevos.", field='purpose', source_field=None, row_count=None, details={'field': 'purpose', 'n_added': 3, 'added_values_sample': ['Estudio', 'Otro', 'Trabajo'], 'added_values_total': 3, 'policy': 'extendable_non_strict', 'action': 'extended_domain'}),
 Issue(level='warning', code='IMP.TEMPORAL.TIER_LIMITED', message='El dataset fue importado con capacidad temporal limitada (tier_3); algunas operaciones posteriores quedarán restringidas.', field=None, source_field=None, row_count=None, details={'temporal_tier': 'tier_3', 'fields_present': [], 'not